# GPT1: Improving Language Understanding by Generative Pre-Training

It demonstrates that large gains on diverse tasks such as textual entailment, question answering, semantic similarity assessment, 
and document classification can be realized by generative pre-training of a language model on a diverse corpus of unlabeled text, 
followed by discriminative fine-tuning on each specific task.

# Import Libraries

In [7]:
import datasets
import re
import ftfy
import os

# Setting import global variables

In [8]:
cpu_cores = os.cpu_count()

# Generative Pre-Training

## Data Prep

### Loading saving and reloading the dataset
We use the BooksCorpus dataset for training the language model. It contains over 7,000 unique unpublished 
books from a variety of genres including Adventure, Fantasy, and Romance. Crucially, it contains long stretches of contiguous 
text, which allows the generative model to learn to condition on long-range information.

In [3]:
# bookscorpus_dataset = datasets.load_dataset("bookcorpus")["train"]
# bookscorpus_dataset.save_to_disk("datasets/bookscorpus")
bookscorpus_dataset = datasets.load_from_disk("datasets/bookscorpus")
len(bookscorpus_dataset), bookscorpus_dataset[0]

(74004228,
 {'text': 'usually , he would be tearing around the living room , playing with his toys .'})

### Cleaning, Saving and Loading cleaned dataset
We use the ftfy library to clean the raw text in BooksCorpus, standardize some punctuation and
whitespace, and use the spaCy tokenizer

In [6]:
def clean_text(sentence):
    """
    - uses ftfy library to clean the raw text
    - fixes some issues the spacy tokenizer had on books corpus
    - also does some whitespace standardization
    """
    # print(sentence)
    text = str(sentence["text"])
    text = ftfy.fix_text(text)
    text = text.replace('—', '-')
    text = text.replace('–', '-')
    text = text.replace('―', '-')
    text = text.replace('…', '...')
    text = text.replace('´', "'")
    text = re.sub('''(-+|~+|!+|"+|;+|\?+|\++|,+|\)+|\(+|\\+|\/+|\*+|\[+|\]+|}+|{+|\|+|_+)''', r' \1 ', text)
    text = re.sub('\s*\n\s*', ' \n ', text)
    text = re.sub('[^\S\n]+', ' ', text)
    text = text.strip()
    sentence["text"] = text
    return sentence

In [ ]:
# bookscorpus_dataset = bookscorpus_dataset.map(clean_text,num_proc=cpu_cores)
# bookscorpus_dataset.save_to_disk("datasets/bookscorpus_cleaned")
bookscorpus_dataset = datasets.load_from_disk("datasets/bookscorpus_cleaned")["text"]